# Lesson 6: Spectral & Time–Frequency Analysis — FFT, Wavelet Transform, EMD, and EEMD

**Course:** TTK4260 — Multivariate Data Analysis & Machine Learning 
**Author:** Prof. Adil Rasheed, NTNU

---

## Learning Objectives

By the end of this lesson you should be able to:

1. **Explain** why the Fourier Transform alone is insufficient for analysing non-stationary signals.
2. **Apply** the FFT to identify dominant frequencies in a stationary signal.
3. **Use** the Continuous Wavelet Transform (CWT) to obtain a *time–frequency* representation.
4. **Decompose** a signal into Intrinsic Mode Functions (IMFs) using EMD and EEMD.
5. **Compare** the strengths and limitations of FFT, WT, EMD, and EEMD on the same synthetic signal.

---

## Conceptual Overview

| Method | Domain | Stationary signals | Non-stationary signals | Adaptive? | Key limitation |
|--------|--------|-------------------|----------------------|-----------|----------------|
| **FFT** | Frequency only | Excellent | Poor — cannot localise events in time | No | Assumes global stationarity |
| **Wavelet Transform** | Time–frequency | Good | Good — adjustable time–frequency resolution | Partially (choice of mother wavelet) | Resolution governed by Heisenberg uncertainty |
| **EMD** | Time (IMFs) | Good | Good — fully data-driven | Yes | Mode mixing; sensitive to noise |
| **EEMD** | Time (IMFs) | Good | Better than EMD | Yes | Computationally heavier; residual noise |

### The core trade-off

- **FFT** gives you *perfect frequency resolution* but *zero time resolution* — you know **which** frequencies are present, but not **when** they occur.
- **Wavelets** compromise: they give *some* of both, bounded by the Heisenberg uncertainty principle ($\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$).
- **EMD/EEMD** sidestep fixed basis functions entirely — they let the data *define its own* oscillatory components (IMFs).

---
## 1. Synthetic Test Signal

We build a signal with **known components** so we can later verify whether each method recovers them correctly.

The signal $y(t)$ is composed of:

$$y(t) = \underbrace{\sin\!\left(\frac{2\pi t}{12}\right)}_{\text{seasonal, period 12}} + \underbrace{\sin\!\left(\frac{2\pi t}{60}\right)}_{\text{cyclical, period 60}}$$

We then inject **localised transient signatures** — short bursts that appear only during limited time intervals. These are the features that **FFT will struggle with** but wavelets and EMD should capture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # Reproducibility

# --- Signal parameters ---
n = 1000                         # Number of samples
t = np.arange(1, n + 1)          # Time index

# --- Deterministic components (known ground truth) ---
seasonal = np.sin(2 * np.pi * t / 12)   # Period = 12 samples  → frequency ≈ 0.0833
cyclical = np.sin(2 * np.pi * t / 60)   # Period = 60 samples  → frequency ≈ 0.0167
y = seasonal + cyclical                  # Clean, stationary base signal

print(f"Signal length     : {n} samples")
print(f"Seasonal frequency: {1/12:.4f} cycles/sample  (period = 12)")
print(f"Cyclical frequency: {1/60:.4f} cycles/sample  (period = 60)")

### 1.1 Adding localised transient signatures

Real-world signals often contain **non-stationary events**: a machine fault pulse, an ocean wave anomaly, a cardiac arrhythmia. We simulate this by injecting short random bursts at random positions.

**Why this matters:** A purely stationary signal (sum of fixed sinusoids) is trivially handled by FFT. The transients break stationarity and create the challenge that motivates wavelets and EMD.

In [ ]:
# --- Transient signature parameters ---
signature_size = 100       # Duration of each transient (samples)
n_signatures = 2           # Number of transient events

# Random amplitude scaling and offsets for each signature
sigmas = np.random.randn(n_signatures) * 5
mus = np.random.randn(n_signatures)

# Random start positions (ensure signatures fit within the signal)
t_signatures = []
for _ in range(n_signatures):
    start = int(np.random.uniform(0, n - signature_size))
    t_signatures.append(range(start, start + signature_size))

# Each signature is a random amplitude burst with an offset
signatures = [s * np.random.rand(signature_size) + m
              for s, m in zip(sigmas, mus)]

# Inject signatures into the clean signal
y_with_signatures = y.copy()
for t_s, s in zip(t_signatures, signatures):
    y_with_signatures[t_s] += s

print(f"Injected {n_signatures} transient signatures of length {signature_size}")
for i, t_s in enumerate(t_signatures):
    print(f"  Signature {i+1}: samples {t_s[0]}–{t_s[-1]}, amplitude σ={sigmas[i]:.2f}")

In [ ]:
# --- Visualise the original and modified signals ---
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y, label="Clean signal (seasonal + cyclical)", alpha=0.7)
ax.plot(y_with_signatures, label="Signal with transient signatures", alpha=0.9)

# Highlight the injected signature regions
offset = -5
for i, (t_s, s) in enumerate(zip(t_signatures, signatures)):
    ax.plot(t_s, s + offset, linewidth=2, label=f"Signature {i+1} (shifted down)")
    ax.axvspan(t_s[0], t_s[-1], alpha=0.1, color='red')

ax.set_xlabel("Sample index", fontsize=13)
ax.set_ylabel("Amplitude", fontsize=13)
ax.set_title("Synthetic Test Signal: Stationary Base + Localised Transients", fontsize=14)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 2. Fast Fourier Transform (FFT)

### 2.1 Theory recap

The Discrete Fourier Transform decomposes a signal into a sum of complex exponentials:

$$X[k] = \sum_{n=0}^{N-1} x[n] \, e^{-j 2\pi k n / N}, \qquad k = 0, 1, \ldots, N-1$$

The FFT is simply an *efficient algorithm* for computing this ($O(N \log N)$ vs. $O(N^2)$).

**Key properties:**
- **Frequency resolution:** $\Delta f = 1/N$ (with unit sampling interval). Longer signals → finer frequency bins.
- **Nyquist frequency:** $f_{\text{max}} = 0.5$ (for unit sampling). Frequencies above this alias.
- **Global analysis:** The transform uses *all* $N$ samples — it tells you *which* frequencies are present *somewhere* in the signal, but not *when* they appear.

### 2.2 What to expect

We should see **two clear peaks** at $f = 1/12 \approx 0.083$ and $f = 1/60 \approx 0.017$. The transient signatures will add **broadband spectral leakage** but their *temporal location* will be invisible in the spectrum.

In [ ]:
from scipy.fft import fft, fftfreq, fftshift

# --- Compute FFT ---
freqs = fftshift(fftfreq(n, d=1))       # Frequency axis (cycles/sample)
Y_clean = fftshift(fft(y))               # FFT of clean signal
Y_signatures = fftshift(fft(y_with_signatures))  # FFT of signal with transients

# We only plot the positive-frequency half
pos = freqs >= 0

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Clean signal FFT ---
axes[0].plot(freqs[pos], (1/n) * np.abs(Y_clean[pos]), color='steelblue')
axes[0].set_title("FFT of Clean Signal (no transients)", fontsize=13)
axes[0].set_xlabel("Frequency (cycles/sample)")
axes[0].set_ylabel("Normalised amplitude")
axes[0].axvline(1/12, color='red', linestyle='--', alpha=0.7, label=f'f = 1/12 ≈ {1/12:.4f}')
axes[0].axvline(1/60, color='green', linestyle='--', alpha=0.7, label=f'f = 1/60 ≈ {1/60:.4f}')
axes[0].legend()

# --- Signal with transients FFT ---
axes[1].plot(freqs[pos], (1/n) * np.abs(Y_signatures[pos]), color='coral')
axes[1].set_title("FFT of Signal with Transients", fontsize=13)
axes[1].set_xlabel("Frequency (cycles/sample)")
axes[1].set_ylabel("Normalised amplitude")
axes[1].axvline(1/12, color='red', linestyle='--', alpha=0.7, label=f'f = 1/12')
axes[1].axvline(1/60, color='green', linestyle='--', alpha=0.7, label=f'f = 1/60')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Observation: Both spectra show the two dominant peaks, but the transient")
print("signatures add broadband energy across all frequencies. Crucially, the FFT")
print("CANNOT tell us WHERE in time those transients occurred.")

### 2.3 Discussion: Limitations of FFT

| Strength | Limitation |
|----------|------------|
| Computationally efficient ($O(N \log N)$) | No time localisation — a 1-second pulse and a 1000-second tone at the same frequency look identical |
| Perfect for **stationary** signals | Non-stationary features "smear" energy across frequency bins |
| Well-understood mathematical properties | Requires windowing to reduce spectral leakage; window choice introduces bias |

> **Key takeaway:** FFT answers *"What frequencies are present?"* but **not** *"When do they occur?"*

### ✏️ Task 1

1. Verify that the FFT peaks correspond to the known frequencies ($1/12$ and $1/60$).
2. Uncomment `trend` and `irregular` components in the signal definition (Cell 1). How does the FFT spectrum change? Why?
3. Try windowing the signal with a Hann window before FFT: `y_windowed = y * np.hanning(n)`. What happens to the spectral leakage?

---
## 3. Continuous Wavelet Transform (CWT)

### 3.1 Theory recap

The CWT correlates the signal with a family of **dilated and translated** wavelets:

$$W(a, b) = \frac{1}{\sqrt{a}} \int_{-\infty}^{\infty} x(t) \, \psi^*\!\left(\frac{t - b}{a}\right) dt$$

where:
- $\psi(t)$ is the **mother wavelet** (e.g., complex Morlet, Daubechies, Mexican hat)
- $a > 0$ is the **scale** (inversely related to frequency: large $a$ → low frequency)
- $b$ is the **translation** (time position)

**Key insight:** At each $(a, b)$ pair, the CWT tells you how much energy the signal has *at that scale* and *at that time*. This is the **time–frequency representation** that FFT lacks.

### 3.2 The Heisenberg trade-off

Wavelets cannot achieve perfect resolution in both time and frequency simultaneously:

- **High frequency (small scale $a$):** Good time resolution, poor frequency resolution.
- **Low frequency (large scale $a$):** Good frequency resolution, poor time resolution.

This is a fundamental property of *all* linear time–frequency representations, not a limitation of a particular wavelet.

### 3.3 What to expect

The scalogram should show:
- **Persistent horizontal bands** at periods 12 and 60 (the stationary sinusoids).
- **Localised bright regions** where the transient signatures occur — this is what FFT misses.

In [ ]:
import pywt
import matplotlib.gridspec as gridspec

# --- Helper functions ---

def plot_signal_and_average(ax, time, signal, avg_window=5):
    """Plot the raw signal with a moving average overlay."""
    # Simple block averaging
    n_blocks = len(signal) // avg_window
    sig_trimmed = signal[:n_blocks * avg_window].reshape(-1, avg_window)
    time_trimmed = time[:n_blocks * avg_window].reshape(-1, avg_window)[:, 0]
    sig_avg = sig_trimmed.mean(axis=1)
    
    ax.plot(time, signal, alpha=0.6, label='Signal')
    ax.plot(time_trimmed, sig_avg, linewidth=2, label=f'Block average (n={avg_window})')
    ax.set_xlim([time[0], time[-1]])
    ax.set_ylabel('Amplitude')
    ax.set_title('Signal + Time Average')
    ax.legend(loc='upper right')


def plot_scalogram(ax, time, signal, scales, wavelet='cmor1.5-1.0',
                   cmap='seismic', title='', ylabel='', xlabel=''):
    """Compute and plot the CWT scalogram (log₂ power)."""
    dt = time[1] - time[0]
    coefficients, frequencies = pywt.cwt(signal, scales, wavelet, dt)
    power = np.abs(coefficients) ** 2
    period = 1.0 / frequencies
    
    # Contour levels in log₂ space
    levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8]
    im = ax.contourf(time, np.log2(period), np.log2(power),
                     np.log2(levels), extend='both', cmap=cmap)
    
    ax.set_title(title, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_xlabel(xlabel, fontsize=13)
    
    # Y-axis: show actual period values
    yticks = 2 ** np.arange(np.ceil(np.log2(period.min())),
                            np.ceil(np.log2(period.max())))
    ax.set_yticks(np.log2(yticks))
    ax.set_yticklabels([f'{int(yt)}' for yt in yticks])
    ax.invert_yaxis()
    ylim = ax.get_ylim()
    ax.set_ylim(ylim[0], -1)
    
    return yticks, ylim


def plot_fft_vertical(ax, time, signal, yticks, ylim):
    """Plot FFT power vertically alongside the scalogram."""
    dt = time[1] - time[0]
    N = len(signal)
    f_values = np.linspace(0, 0.5 / dt, N // 2)
    fft_values = (2.0 / N) * np.abs(fft(signal)[:N // 2])
    
    scales = 1.0 / (f_values + 1e-10)  # Avoid division by zero
    ax.plot(fft_values, np.log2(scales), 'r-', linewidth=0.8, label='FFT amplitude')
    ax.set_yticks(np.log2(yticks))
    ax.set_yticklabels([])
    ax.invert_yaxis()
    ax.set_ylim(ylim[0], -1)
    ax.set_xlabel('Amplitude')
    ax.legend(fontsize=8)

In [ ]:
from scipy.fftpack import fft as fft_pack  # for the vertical FFT plot

# --- Compute and display the CWT scalogram ---
scales = np.arange(1, 128)

fig = plt.figure(figsize=(14, 10))
spec = gridspec.GridSpec(ncols=6, nrows=6, figure=fig)

# Top panel: signal
top_ax = fig.add_subplot(spec[0, 0:5])
plot_signal_and_average(top_ax, t, y_with_signatures, avg_window=5)

# Bottom-left: scalogram
scalogram_ax = fig.add_subplot(spec[1:, 0:5])
yticks, ylim = plot_scalogram(
    scalogram_ax, t, y_with_signatures, scales,
    title='CWT Scalogram (log₂ power)',
    ylabel='Period (samples)', xlabel='Sample index'
)

# Annotate expected periods
scalogram_ax.axhline(np.log2(12), color='lime', linestyle='--', alpha=0.7, linewidth=1)
scalogram_ax.axhline(np.log2(60), color='lime', linestyle='--', alpha=0.7, linewidth=1)
scalogram_ax.text(20, np.log2(12) - 0.3, 'Period=12', color='lime', fontsize=9)
scalogram_ax.text(20, np.log2(60) - 0.3, 'Period=60', color='lime', fontsize=9)

# Bottom-right: vertical FFT for comparison
fft_ax = fig.add_subplot(spec[1:, 5])
plot_fft_vertical(fft_ax, t, y_with_signatures, yticks, ylim)

plt.tight_layout()
plt.show()

print("Observation: The scalogram reveals BOTH the persistent sinusoidal components")
print("(horizontal bands at periods 12 and 60) AND the localised transient events")
print("(bright patches at specific time locations). This is the key advantage over FFT.")

### 3.4 Discussion: Strengths & limitations of Wavelet Transform

| Strength | Limitation |
|----------|------------|
| Simultaneous time and frequency information | Must choose a mother wavelet *a priori* — different wavelets emphasise different features |
| Multi-resolution: automatically adapts resolution to frequency | Heisenberg trade-off limits joint resolution |
| Edge effects well-characterised via the cone of influence | Computationally more expensive than FFT |
| Works well for signals with transient events | Interpretation of the scalogram requires experience |

> **Key takeaway:** WT answers *"What frequencies are present, **and when**?"* — but requires you to choose the analysis wavelet.

### ✏️ Task 2

1. Try different mother wavelets: `'morl'` (real Morlet), `'mexh'` (Mexican hat), `'gaus1'`. How does the scalogram change?
2. Change the scale range: `np.arange(1, 256)`. What happens at the low-frequency end?
3. Can you precisely pinpoint the *start and end* of each transient from the scalogram? Compare with the known injection times.

---
## 4. Empirical Mode Decomposition (EMD)

### 4.1 Theory recap

EMD is a **data-driven**, fully adaptive decomposition introduced by Huang et al. (1998). It requires **no predefined basis functions** — the decomposition is determined entirely by the signal itself.

**The sifting process** extracts Intrinsic Mode Functions (IMFs) iteratively:

1. Identify all local maxima and minima of the signal.
2. Construct upper and lower envelopes by cubic spline interpolation.
3. Compute the mean envelope: $m(t) = \frac{e_{\max}(t) + e_{\min}(t)}{2}$
4. Subtract: $h(t) = x(t) - m(t)$
5. Repeat until $h(t)$ satisfies the IMF criteria:
   - Number of extrema and zero crossings differ by at most 1.
   - Mean of upper and lower envelopes is approximately zero.
6. Store this IMF, subtract it from the signal, and repeat on the residual.

The result is: $x(t) = \sum_{k=1}^{K} \text{IMF}_k(t) + r(t)$, where $r(t)$ is the residual (trend).

### 4.2 What to expect

- The first few IMFs should capture **high-frequency** oscillations (including transient effects).
- Later IMFs should isolate the **seasonal** (period 12) and **cyclical** (period 60) components.
- The last IMF or residual captures any low-frequency trend.

**Known issue — mode mixing:** EMD can sometimes assign parts of different physical phenomena to the same IMF, especially when the signal contains intermittent components. This motivates EEMD (Section 5).

In [ ]:
from PyEMD import EMD, Visualisation

# --- Perform EMD ---
emd = EMD()
IMFs_emd = emd(y_with_signatures)

print(f"EMD produced {IMFs_emd.shape[0]} IMFs")
print(f"Signal can be reconstructed as: signal ≈ Σ IMFs")
print(f"Reconstruction error: {np.max(np.abs(y_with_signatures - IMFs_emd.sum(axis=0))):.2e}")

# --- Plot IMFs ---
fig, axes = plt.subplots(IMFs_emd.shape[0] + 1, 1,
                         figsize=(12, 2.5 * (IMFs_emd.shape[0] + 1)))

axes[0].plot(y_with_signatures, color='steelblue')
axes[0].set_title('Original Signal', fontsize=12)
axes[0].set_ylabel('Amplitude')

for i, imf in enumerate(IMFs_emd):
    axes[i + 1].plot(imf, color='coral')
    axes[i + 1].set_title(f'IMF {i + 1}', fontsize=12)
    axes[i + 1].set_ylabel('Amplitude')

axes[-1].set_xlabel('Sample index')
plt.tight_layout()
plt.show()

### 4.3 Frequency content of each IMF

By taking the FFT of each IMF, we can verify whether EMD has successfully separated the known frequency components. We overlay the FFT of the *true* seasonal and cyclical components (dashed lines) for comparison.

In [ ]:
# --- FFT of each IMF vs. ground truth ---
from scipy.fft import fft as scipy_fft

# Ground truth FFTs
seasonal_fft = fftshift(scipy_fft(seasonal))
cyclical_fft = fftshift(scipy_fft(cyclical))

n_imfs = IMFs_emd.shape[0]
fig, axes = plt.subplots(n_imfs, 1, figsize=(10, 2.8 * n_imfs))
if n_imfs == 1:
    axes = [axes]

for i, imf in enumerate(IMFs_emd):
    imf_fft = fftshift(scipy_fft(imf))
    
    axes[i].plot(freqs[pos], (1/n) * np.abs(imf_fft[pos]),
                label=f'IMF {i+1}', color='steelblue', linewidth=1.5)
    axes[i].plot(freqs[pos], (1/n) * np.abs(seasonal_fft[pos]),
                label='Seasonal (f=1/12)', color='green', alpha=0.4, linestyle='--')
    axes[i].plot(freqs[pos], (1/n) * np.abs(cyclical_fft[pos]),
                label='Cyclical (f=1/60)', color='red', alpha=0.4, linestyle='--')
    axes[i].set_title(f'FFT of IMF {i+1}', fontsize=11)
    axes[i].set_ylabel('Amplitude')
    axes[i].legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Frequency (cycles/sample)')
plt.tight_layout()
plt.show()

print("Look for IMFs whose FFT peaks align with the green (seasonal) and red (cyclical) dashed lines.")
print("IMFs capturing transient signatures will show broader spectral content.")

---
## 5. Ensemble Empirical Mode Decomposition (EEMD)

### 5.1 Motivation: fixing mode mixing

EMD's main weakness is **mode mixing** — a single IMF may contain oscillations of wildly different time scales, or a single physical mode may be split across multiple IMFs.

**EEMD** (Wu & Huang, 2009) addresses this by:

1. Adding white noise to the signal.
2. Performing EMD on the noisy version.
3. Repeating steps 1–2 many times (an *ensemble*).
4. Averaging the corresponding IMFs across all ensemble members.

**Why it works:** The added noise populates the full time–frequency space uniformly, forcing EMD to find consistent scale separations. Noise cancels out in the ensemble average, while the true signal components reinforce.

**Parameters:**
- `trials` (ensemble size): More trials → cleaner separation, but slower. Typical: 50–500.
- `noise_width`: Standard deviation of added noise as a fraction of signal std. Typical: 0.02–0.2.

In [ ]:
from PyEMD import EEMD

# --- Perform EEMD ---
eemd = EEMD(trials=100, noise_width=0.05)
IMFs_eemd = eemd(y_with_signatures)

print(f"EEMD produced {IMFs_eemd.shape[0]} IMFs (ensemble of 100 trials)")
print(f"Reconstruction error: {np.max(np.abs(y_with_signatures - IMFs_eemd.sum(axis=0))):.2e}")

# --- Plot EEMD IMFs ---
n_eemd = IMFs_eemd.shape[0]
fig, axes = plt.subplots(n_eemd + 1, 1,
                         figsize=(12, 2.5 * (n_eemd + 1)))

axes[0].plot(y_with_signatures, color='steelblue')
axes[0].set_title('Original Signal', fontsize=12)
axes[0].set_ylabel('Amplitude')

for i, imf in enumerate(IMFs_eemd):
    axes[i + 1].plot(imf, color='teal')
    axes[i + 1].set_title(f'EEMD IMF {i + 1}', fontsize=12)
    axes[i + 1].set_ylabel('Amplitude')

axes[-1].set_xlabel('Sample index')
plt.tight_layout()
plt.show()

### 5.2 Frequency content of EEMD IMFs

In [ ]:
# --- FFT of EEMD IMFs vs. ground truth ---
n_eemd = IMFs_eemd.shape[0]
fig, axes = plt.subplots(n_eemd, 1, figsize=(10, 2.8 * n_eemd))
if n_eemd == 1:
    axes = [axes]

for i, imf in enumerate(IMFs_eemd):
    imf_fft = fftshift(scipy_fft(imf))
    
    axes[i].plot(freqs[pos], (1/n) * np.abs(imf_fft[pos]),
                label=f'EEMD IMF {i+1}', color='teal', linewidth=1.5)
    axes[i].plot(freqs[pos], (1/n) * np.abs(seasonal_fft[pos]),
                label='Seasonal (f=1/12)', color='green', alpha=0.4, linestyle='--')
    axes[i].plot(freqs[pos], (1/n) * np.abs(cyclical_fft[pos]),
                label='Cyclical (f=1/60)', color='red', alpha=0.4, linestyle='--')
    axes[i].set_title(f'FFT of EEMD IMF {i+1}', fontsize=11)
    axes[i].set_ylabel('Amplitude')
    axes[i].legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Frequency (cycles/sample)')
plt.tight_layout()
plt.show()

print("Compare with the EMD IMF spectra above — EEMD should show cleaner separation")
print("of the frequency components, with less mode mixing.")

---
## 6. Side-by-Side Comparison

Let us now directly compare what each method reveals about our test signal.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (a) FFT — frequency domain only
axes[0, 0].plot(freqs[pos], (1/n) * np.abs(Y_signatures[pos]), color='steelblue')
axes[0, 0].axvline(1/12, color='red', linestyle='--', alpha=0.6)
axes[0, 0].axvline(1/60, color='green', linestyle='--', alpha=0.6)
axes[0, 0].set_title('(a) FFT — Frequency Domain', fontsize=12)
axes[0, 0].set_xlabel('Frequency')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].text(0.05, 0.85, 'No time info!', transform=axes[0, 0].transAxes,
               fontsize=11, color='red', fontweight='bold')

# (b) CWT Scalogram — time-frequency
coefficients, frequencies = pywt.cwt(y_with_signatures, scales, 'cmor1.5-1.0', 1)
power = np.abs(coefficients) ** 2
period = 1.0 / frequencies
levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8]
axes[0, 1].contourf(t, np.log2(period), np.log2(power),
                    np.log2(levels), extend='both', cmap='seismic')
axes[0, 1].set_title('(b) CWT Scalogram — Time–Frequency', fontsize=12)
axes[0, 1].set_xlabel('Sample index')
axes[0, 1].set_ylabel('Period')
axes[0, 1].invert_yaxis()
yticks_comp = 2 ** np.arange(np.ceil(np.log2(period.min())),
                             np.ceil(np.log2(period.max())))
axes[0, 1].set_yticks(np.log2(yticks_comp))
axes[0, 1].set_yticklabels([f'{int(yt)}' for yt in yticks_comp])

# (c) EMD — first 3 IMFs
n_show = min(3, IMFs_emd.shape[0])
for i in range(n_show):
    axes[1, 0].plot(IMFs_emd[i] + i * 4, label=f'IMF {i+1}')
axes[1, 0].set_title(f'(c) EMD — First {n_show} IMFs (offset for clarity)', fontsize=12)
axes[1, 0].set_xlabel('Sample index')
axes[1, 0].legend(loc='upper right', fontsize=9)
axes[1, 0].set_ylabel('Amplitude (offset)')

# (d) EEMD — first 3 IMFs
n_show = min(3, IMFs_eemd.shape[0])
for i in range(n_show):
    axes[1, 1].plot(IMFs_eemd[i] + i * 4, label=f'IMF {i+1}')
axes[1, 1].set_title(f'(d) EEMD — First {n_show} IMFs (offset for clarity)', fontsize=12)
axes[1, 1].set_xlabel('Sample index')
axes[1, 1].legend(loc='upper right', fontsize=9)
axes[1, 1].set_ylabel('Amplitude (offset)')

plt.suptitle('Comparison of Spectral & Time–Frequency Methods', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Summary & Key Takeaways

### When to use which method

| Scenario | Recommended method | Reasoning |
|----------|-------------------|------------|
| Stationary signal, need frequency content | **FFT** | Fast, exact, well-understood |
| Non-stationary signal, need to know *when* frequency content changes | **Wavelet Transform** | Time–frequency representation |
| Non-stationary signal, want a fully data-driven decomposition | **EMD** | No basis function choice needed |
| EMD shows mode mixing or signal is noisy | **EEMD** | Ensemble averaging reduces mode mixing |
| Need to combine time localisation with frequency analysis | **EMD/EEMD + FFT** on each IMF | Best of both worlds |

### The big picture

These methods form a **progression of increasing adaptivity**:

$$\text{FFT} \xrightarrow{\text{add time localisation}} \text{WT} \xrightarrow{\text{remove basis choice}} \text{EMD} \xrightarrow{\text{fix mode mixing}} \text{EEMD}$$

Each step trades some mathematical elegance for more adaptivity to the data. In practice, **the best method depends on what you know about the signal** and **what question you are trying to answer**.

### ✏️ Task 3 (synthesis)

1. Add Gaussian noise ($\sigma = 0.5$) to the signal. How does each method's output degrade?
2. Increase `signature_size` to 500. Does this help or hurt FFT? Why?
3. Try `EEMD(trials=10)` vs `EEMD(trials=500)`. What is the effect on mode mixing and computation time?
4. **Challenge:** Apply these methods to a real-world dataset (e.g., ECG, vibration, or financial time series) and discuss which method gives the most useful decomposition.

---

### References

- Huang, N.E. et al. (1998). *The empirical mode decomposition and the Hilbert spectrum for nonlinear and non-stationary time series analysis.* Proc. Royal Society A, 454, 903–995.
- Wu, Z. & Huang, N.E. (2009). *Ensemble empirical mode decomposition: a noise-assisted data analysis method.* Advances in Adaptive Data Analysis, 1(1), 1–41.
- Torrence, C. & Compo, G.P. (1998). *A practical guide to wavelet analysis.* Bulletin of the AMS, 79(1), 61–78.